# AgTech: Automação de Precisão — Etapa 3

## Redes Neurais Artificiais + Sistema Especialista + Gemini API

Este notebook implementa a **Etapa 3** do projeto AgTech. A abordagem escolhida foi a **Opção A — Redes Neurais Artificiais (RNA)**.

A RNA prevê a **umidade do solo na próxima hora** a partir dos sensores definidos na Etapa 1. Esse resultado alimenta o **Sistema Especialista** da Etapa 2, que decide os atuadores de irrigação e manejo de pragas. Por fim, a **API do Gemini** gera uma análise interpretativa em linguagem natural.

Fluxo:

```text
Dados dos sensores → RNA → Previsão → Sistema Especialista → Atuadores → Gemini API
```

## 1. Instalação das bibliotecas

No Google Colab, execute esta célula antes das demais. A biblioteca `google-genai` é usada para a integração com a API Gemini.

In [ ]:
!pip install -q -U google-genai scikit-learn pandas numpy matplotlib

## 2. Importação das bibliotecas

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 3. Geração de dados simulados dos sensores

Em um protótipo real, estes dados viriam de sensores embarcados no drone, robô agrícola ou estação de campo. Para fins acadêmicos, os dados são simulados de forma controlada.

Sensores usados:

- umidade do solo atual;
- temperatura;
- umidade do ar;
- velocidade do vento;
- chuva prevista;
- evapotranspiração estimada;
- confiança da visão computacional para presença de praga;
- dano foliar estimado pela imagem.

A variável que a RNA deve aprender é:

```text
umidade_solo_prox_hora_pct
```

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n = 1600

umidade_solo_atual = rng.uniform(15, 85, n)
temperatura = rng.uniform(18, 38, n)
umidade_ar = rng.uniform(30, 95, n)
vento = rng.uniform(0, 20, n)
chuva_prevista = rng.gamma(shape=1.3, scale=4.5, size=n).clip(0, 30)

# Evapotranspiração aproximada para fins didáticos.
evapotranspiracao = (
    0.09 * temperatura +
    0.045 * vento -
    0.025 * umidade_ar +
    rng.normal(0, 0.15, n)
).clip(0.2, 5.0)

# Saída simulada de uma visão computacional/RNA de imagem.
confianca_praga_visao = (
    0.35 +
    0.015 * (temperatura - 24) +
    0.003 * (umidade_ar - 55) +
    rng.normal(0, 0.18, n)
).clip(0, 0.99)

dano_foliar = (confianca_praga_visao * 38 + rng.normal(0, 6, n)).clip(0, 60)

# Alvo da RNA: umidade do solo na próxima hora.
umidade_solo_prox_hora = (
    umidade_solo_atual
    - 1.9 * evapotranspiracao
    - 0.055 * vento
    + 0.82 * chuva_prevista
    - 0.025 * dano_foliar
    + rng.normal(0, 2.4, n)
).clip(5, 95)

df = pd.DataFrame({
    'umidade_solo_atual_pct': umidade_solo_atual.round(2),
    'temperatura_c': temperatura.round(2),
    'umidade_ar_pct': umidade_ar.round(2),
    'vento_kmh': vento.round(2),
    'chuva_prevista_mm': chuva_prevista.round(2),
    'evapotranspiracao_mm_h': evapotranspiracao.round(2),
    'confianca_praga_visao': confianca_praga_visao.round(3),
    'dano_foliar_pct': dano_foliar.round(2),
    'umidade_solo_prox_hora_pct': umidade_solo_prox_hora.round(2)
})

df.head()

## 4. Treinamento da Rede Neural Artificial

A RNA usada é uma `MLPRegressor`, uma rede neural multilayer perceptron para problemas de regressão.

A rede aprende a prever a umidade do solo na próxima hora. Essa previsão será usada pelo Sistema Especialista para tomar uma decisão mais antecipada.

In [ ]:
features = [
    'umidade_solo_atual_pct',
    'temperatura_c',
    'umidade_ar_pct',
    'vento_kmh',
    'chuva_prevista_mm',
    'evapotranspiracao_mm_h',
    'confianca_praga_visao',
    'dano_foliar_pct'
]

target = 'umidade_solo_prox_hora_pct'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED
)

modelo_rna = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation='relu',
        solver='adam',
        max_iter=450,
        learning_rate_init=0.008,
        random_state=RANDOM_SEED,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=25
    )
)

modelo_rna.fit(X_train, y_train)

predicoes = modelo_rna.predict(X_test).clip(5, 95)

mae = mean_absolute_error(y_test, predicoes)
rmse = mean_squared_error(y_test, predicoes) ** 0.5
r2 = r2_score(y_test, predicoes)

print(f'MAE: {mae:.3f} pontos percentuais de umidade')
print(f'RMSE: {rmse:.3f} pontos percentuais de umidade')
print(f'R²: {r2:.3f}')

## 5. Gráfico de desempenho: curva de Loss

Este gráfico demonstra que a RNA está aprendendo. A tendência esperada é a redução da perda durante as épocas de treinamento.

In [ ]:
mlp = modelo_rna.named_steps['mlpregressor']
loss = mlp.loss_curve_

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(loss) + 1), loss, marker='o', markersize=2, linewidth=1.5)
plt.title('Evolução da perda da RNA durante o treinamento')
plt.xlabel('Época de treinamento')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Comparação entre valor real e valor previsto

Este gráfico ajuda a verificar se a RNA aproxima corretamente a umidade futura.

In [ ]:
plt.figure(figsize=(7.5, 6))
plt.scatter(y_test, predicoes, alpha=0.55, s=18)
minv, maxv = min(y_test.min(), predicoes.min()), max(y_test.max(), predicoes.max())
plt.plot([minv, maxv], [minv, maxv], linestyle='--')
plt.title('Comparação entre umidade real e prevista pela RNA')
plt.xlabel('Umidade real próxima hora (%)')
plt.ylabel('Umidade prevista pela RNA (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Sistema Especialista da Etapa 2

A decisão técnica continua sendo feita pelo Sistema Especialista. A diferença é que agora ele recebe uma informação aprendida pela RNA: a previsão da umidade do solo na próxima hora.

In [ ]:
def classificar_praga(confianca_praga, dano_foliar):
    if confianca_praga >= 0.85 and dano_foliar >= 25:
        return 'CRÍTICO'
    if confianca_praga >= 0.70 or dano_foliar >= 25:
        return 'ALTO'
    if confianca_praga >= 0.45 or dano_foliar >= 12:
        return 'MÉDIO'
    return 'BAIXO'


def classificar_hidrico(umidade_prevista, temperatura, chuva_prevista):
    if umidade_prevista < 25 and temperatura >= 30:
        return 'DÉFICIT CRÍTICO'
    if umidade_prevista < 35:
        return 'DÉFICIT'
    if umidade_prevista > 78 or (chuva_prevista >= 18 and umidade_prevista >= 60):
        return 'EXCESSO HÍDRICO'
    return 'ADEQUADO'


def sistema_especialista(amostra, umidade_para_decisao):
    risco_praga = classificar_praga(
        float(amostra['confianca_praga_visao']),
        float(amostra['dano_foliar_pct'])
    )

    risco_hidrico = classificar_hidrico(
        float(umidade_para_decisao),
        float(amostra['temperatura_c']),
        float(amostra['chuva_prevista_mm'])
    )

    if risco_hidrico == 'DÉFICIT CRÍTICO':
        irrigacao = 'ACIONAR irrigação em nível ALTO por setor prioritário'
        vazao = 100
    elif risco_hidrico == 'DÉFICIT':
        irrigacao = 'ACIONAR irrigação em nível MÉDIO e reavaliar em 1 hora'
        vazao = 60
    elif risco_hidrico == 'EXCESSO HÍDRICO':
        irrigacao = 'BLOQUEAR irrigação e emitir alerta de excesso de água'
        vazao = 0
    else:
        irrigacao = 'MANTER irrigação desligada; solo em faixa adequada'
        vazao = 0

    if risco_praga == 'CRÍTICO':
        manejo_praga = 'Acionar drone/robô para pulverização localizada e alerta ao agrônomo'
    elif risco_praga == 'ALTO':
        manejo_praga = 'Realizar inspeção direcionada e controle localizado preventivo'
    elif risco_praga == 'MÉDIO':
        manejo_praga = 'Aumentar frequência de monitoramento por imagem'
    else:
        manejo_praga = 'Manter monitoramento de rotina'

    return {
        'risco_praga': risco_praga,
        'risco_hidrico': risco_hidrico,
        'acao_irrigacao': irrigacao,
        'vazao_irrigacao_pct': vazao,
        'acao_manejo_praga': manejo_praga
    }

## 8. Integração: Dados → Aprendizado → Sistema Especialista

Neste cenário, comparamos a decisão antes e depois da RNA:

- **Antes:** o Sistema Especialista usa apenas a umidade atual.
- **Depois:** o Sistema Especialista usa a umidade prevista pela RNA.

In [ ]:
cenario = pd.DataFrame([{
    'umidade_solo_atual_pct': 38.0,
    'temperatura_c': 34.5,
    'umidade_ar_pct': 43.0,
    'vento_kmh': 14.5,
    'chuva_prevista_mm': 0.5,
    'evapotranspiracao_mm_h': 3.9,
    'confianca_praga_visao': 0.82,
    'dano_foliar_pct': 28.0
}])

umidade_prevista_rna = float(np.clip(modelo_rna.predict(cenario[features])[0], 5, 95))

decisao_antes = sistema_especialista(
    cenario.iloc[0],
    cenario.iloc[0]['umidade_solo_atual_pct']
)

decisao_depois = sistema_especialista(
    cenario.iloc[0],
    umidade_prevista_rna
)

print('Umidade atual usada antes do aprendizado:', cenario.iloc[0]['umidade_solo_atual_pct'])
print('Umidade prevista pela RNA para a próxima hora:', round(umidade_prevista_rna, 2))
print('\nDecisão antes do aprendizado:')
print(json.dumps(decisao_antes, ensure_ascii=False, indent=2))
print('\nDecisão depois do aprendizado:')
print(json.dumps(decisao_depois, ensure_ascii=False, indent=2))

## 9. Demonstração visual antes e depois

Este gráfico atende ao critério de demonstração de ajuste, mostrando como a entrada da decisão muda após o aprendizado.

In [ ]:
plt.figure(figsize=(8, 5))
labels = ['Antes: regra usa\numidade atual', 'Depois: regra usa\nprevisão da RNA']
valores = [float(cenario.iloc[0]['umidade_solo_atual_pct']), umidade_prevista_rna]

plt.bar(labels, valores)
plt.axhline(35, linestyle='--', linewidth=1.3, label='limiar de déficit: 35%')
plt.axhline(25, linestyle=':', linewidth=1.3, label='limiar crítico: 25%')
plt.title('Demonstração de ajuste: decisão reativa vs. decisão preditiva')
plt.ylabel('Umidade considerada pelo Sistema Especialista (%)')
plt.ylim(0, 60)
plt.legend()
plt.tight_layout()
plt.show()

## 10. O que a RNA aprendeu?

A técnica abaixo mede a importância aproximada dos sensores usando `permutation_importance`. Isso ajuda a explicar quais entradas mais influenciaram a previsão da umidade futura.

In [ ]:
resultado_importancia = permutation_importance(
    modelo_rna,
    X_test,
    y_test,
    n_repeats=8,
    random_state=RANDOM_SEED,
    scoring='neg_mean_absolute_error'
)

importancias = pd.DataFrame({
    'sensor': features,
    'importancia_media': resultado_importancia.importances_mean
}).sort_values('importancia_media', ascending=False)

importancias

## 11. Integração com a API do Gemini

O Gemini **não decide** tecnicamente a irrigação nem o manejo de pragas. Ele recebe o resultado da RNA e do Sistema Especialista e gera uma explicação em linguagem natural.

Para usar no Colab:

1. Clique no ícone de chave/Secrets no Colab.
2. Crie um segredo chamado `GEMINI_API_KEY`.
3. Cole sua chave da API Gemini.
4. Execute a célula abaixo.

Caso a chave não esteja configurada, o notebook não quebra; ele apenas gera uma análise local de exemplo.

In [ ]:
def obter_gemini_api_key():
    # Primeiro tenta ler de variável de ambiente.
    api_key = os.environ.get('GEMINI_API_KEY')
    if api_key:
        return api_key

    # Depois tenta ler dos Secrets do Google Colab.
    try:
        from google.colab import userdata
        return userdata.get('GEMINI_API_KEY')
    except Exception:
        return None


def gerar_analise_gemini(payload):
    api_key = obter_gemini_api_key()

    prompt = (
        "Você é um especialista em Engenharia de Controle e Automação aplicada à agricultura de precisão.\n\n"
        "Analise o resultado de um sistema AgTech composto por sensores agrícolas, uma Rede Neural Artificial "
        "que prevê a umidade do solo da próxima hora, um Sistema Especialista que decide irrigação e manejo "
        "de pragas, e atuadores como bomba, válvula, drone ou robô agrícola.\n\n"
        "Importante: a decisão técnica já foi tomada pelo Sistema Especialista. Sua função é apenas explicar "
        "em linguagem natural e sugerir alertas estratégicos.\n\n"
        f"Dados do sistema:\n{json.dumps(payload, ensure_ascii=False, indent=2)}\n\n"
        "Gere uma resposta com: explicação do que a RNA aprendeu; comparação entre antes e depois do aprendizado; "
        "justificativa da decisão final; sugestões de melhoria para o campo; alertas estratégicos para o operador."
    )

    if not api_key:
        return (
            "Chave GEMINI_API_KEY não configurada.\n\n"
            "Análise interpretativa local de exemplo:\n"
            "A RNA aprendeu a antecipar a queda da umidade do solo a partir da combinação entre temperatura elevada, "
            "vento, evapotranspiração e baixa previsão de chuva. Antes do aprendizado, o sistema considerava apenas "
            "a umidade atual, o que poderia atrasar a irrigação. Depois do aprendizado, o Sistema Especialista passou "
            "a receber a previsão da próxima hora, permitindo acionar a irrigação de forma preventiva. Como também "
            "existe risco alto de praga, recomenda-se inspeção direcionada e controle localizado."
        )

    from google import genai
    client = genai.Client(api_key=api_key)

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )

    return response.text

## 12. Execução final integrada

Esta célula consolida o fluxo completo da Etapa 3:

```text
Dados → Aprendizado → Lógica da Etapa 2 → Gemini API
```

In [ ]:
payload_final = {
    'abordagem_etapa3': 'Redes Neurais Artificiais - MLPRegressor/Scikit-Learn',
    'objetivo_modelo': 'Prever a umidade do solo da próxima hora para antecipar a decisão de irrigação.',
    'metricas': {
        'MAE_pct_umidade': round(float(mae), 3),
        'RMSE_pct_umidade': round(float(rmse), 3),
        'R2': round(float(r2), 3),
        'epocas_treinadas': len(loss)
    },
    'cenario_teste': cenario.iloc[0].to_dict(),
    'umidade_prevista_rna_pct': round(float(umidade_prevista_rna), 2),
    'decisao_antes_aprendizado': decisao_antes,
    'decisao_depois_aprendizado': decisao_depois,
    'top_sensores_aprendidos': importancias.head(5).to_dict(orient='records')
}

analise = gerar_analise_gemini(payload_final)

print('===== LOG FINAL DO SISTEMA =====')
print(json.dumps(payload_final, ensure_ascii=False, indent=2))

print('\n===== ANÁLISE INTERPRETATIVA DO GEMINI =====')
print(analise)

## 13. Exportação dos logs

Os logs podem ser colocados no GitHub como evidência de funcionamento da integração entre RNA, Sistema Especialista e Gemini.

In [ ]:
with open('log_integracao_etapa3.json', 'w', encoding='utf-8') as arquivo:
    json.dump(payload_final, arquivo, ensure_ascii=False, indent=2)

importancias.to_csv('importancia_sensores_rna.csv', index=False)
df.to_csv('dados_sensores_simulados.csv', index=False)

print('Arquivos exportados:')
print('- log_integracao_etapa3.json')
print('- importancia_sensores_rna.csv')
print('- dados_sensores_simulados.csv')